In [36]:
import os

import numpy as np

from langchain_openai import ChatOpenAI

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from chromadb.utils import embedding_functions

## librería para evaluación 
from langchain_openai import OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

from datasets import Dataset
from ragas.metrics import (
    Faithfulness,
    AnswerCorrectness,
    ResponseRelevancy,
    FactualCorrectness,
)
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

from dotenv import load_dotenv

load_dotenv()

openai_key = os.getenv("OPENAI_KEY")

/var/folders/rp/m4lwqbf514n2ywy4y_md9txr0000gn/T/ipykernel_6316/1511505582.py:21: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/rp/m4lwqbf514n2ywy4y_md9txr0000gn/T/ipykernel_6316/1511505582.py:21: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import (
/var/folders/rp/m4lwqbf514n2ywy4y_md9txr0000gn/T/ipykernel_6316/1511505582.py:21: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (

## Generación RAG simple

In [23]:
# Paso 1) Crear base vectorial
all_txt_files = os.listdir("data")
all_txt_files = [i for i in all_txt_files if "txt" in i]
print(all_txt_files)

client = chromadb.Client()

## Borrar colecciones existentes
for i in range(1,4):
    try:
        client.delete_collection(name=f"all-my-documents-{i}")
    except Exception:
        pass

## Crear clase de embedding compatible entre HF y ChromaDB
class NewEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        self.model = SentenceTransformer(
            "microsoft/harrier-oss-v1-0.6b",
            trust_remote_code=True,
            model_kwargs={"attn_implementation": "eager", "torch_dtype": "bfloat16"},
            tokenizer_kwargs={"padding_side": "left"},  # hiper-parámetros recomendados por autores: https://huggingface.co/nvidia/llama-embed-nemotron-8b
        )

    def __call__(self, input: Documents) -> Embeddings:
        return self.model.encode(input).tolist()

    @staticmethod
    def name() -> str:
        return "harrier-oss-v1-0.6b"

ef = NewEmbeddingFunction()

## Crear colecciones
collection_1 = client.get_or_create_collection(
    name="all-my-documents-1",
    metadata={"hnsw:space": "cosine"} 
)

collection_2 = client.get_or_create_collection(
    name="all-my-documents-2",
    metadata={"hnsw:space": "cosine"},
    embedding_function=ef,
)
ef_openai = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_KEY"],
    model_name="text-embedding-3-large",
)

collection_3 = client.get_or_create_collection(
    name="all-my-documents-3",
    metadata={"hnsw:space": "cosine"},
    embedding_function=ef_openai,
)

## Chunk texto
splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,        # caracteres, no tokens
    chunk_overlap=25,      # solapamiento para no cortar ideas
    separators=["\n\n", "\n", ". ", " ", ""],
)

## Crear colecciones con diferentes EF
total_chunks = 0 
for file in all_txt_files:
    with open(os.path.join("data",file), "r", encoding="utf-8") as f:
        document_i = f.read()
    chunks = splitter.split_text(document_i)
    collection_1.add(
        documents=chunks,
        ids= [f"{file}_{i}" for i in range(len(chunks))]
    )
    total_chunks += len(chunks)

for file in all_txt_files:
    with open(os.path.join("data",file), "r", encoding="utf-8") as f:
        document_i = f.read()
    chunks = splitter.split_text(document_i)
    collection_2.add(
        documents=chunks,
        ids= [f"{file}_{i}" for i in range(len(chunks))]
    )

for file in all_txt_files:
    with open(os.path.join("data",file), "r", encoding="utf-8") as f:
        document_i = f.read()
    chunks = splitter.split_text(document_i)
    collection_3.add(
        documents=chunks,
        ids= [f"{file}_{i}" for i in range(len(chunks))]
    )

# paso 2) conexión a LLM
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5.4-mini"),
    api_key=os.getenv("OPENAI_KEY"),
    temperature=0,
        model_kwargs={
        "reasoning_effort": "medium"
    }
)

response = llm.invoke("What is the capital of France?")
print(f"{response.content=}") 

# Paso 3) RAG function
def rag_simple(human_query, collection, top_k=2):
    retriever_result = collection.query(
        query_texts=human_query,
        n_results=top_k
    )
    augmented_prompt = "<human_query>" + human_query + "</human_query>"
    augmented_prompt += "<retriever_results>"
    retrieved_list = []
    for i in range(len(retriever_result['ids'][0])):
        
        augmented_prompt += "<doc_id>" + retriever_result['ids'][0][i] + "</doc_id>" 
        augmented_prompt += "<chunk>" + retriever_result['documents'][0][i] +  "</chunk>" 
        retrieved_list.append(retriever_result['documents'][0][i])
    augmented_prompt += "</retriever_results>"

    mensajes = [
        SystemMessage("Eres un sistema RAG experto, responde la pregunta del usuario basado en los chunks. " \
        "Tus respuestas son lo más corta posibles y precisas. No parafrases"),
        HumanMessage(augmented_prompt),
    ]
    cadena = llm | StrOutputParser()

    response = cadena.invoke(mensajes)

    return response, retrieved_list

print(f"{total_chunks=}")

['08_manual_uso_lr200.txt', '02_producto_lr200_specs.txt', '06_garantias.txt', '10_comparativa_competencia.txt', '03_politica_devoluciones_2023.txt', '07_soporte_contactos.txt', '05_acta_reunion_q3_2024.txt', '01_producto_lr100_specs.txt', '04_politica_devoluciones_2024.txt', '09_incidencias_conocidas.txt']


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 19711.27it/s]
/Users/user/miniforge3/envs/lang/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3688: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


response.content='The capital of France is **Paris**.'
total_chunks=53


## Evaluación con LLM como juez usando RAGAS


### 1. `Faithfulness()`
**¿La respuesta es fiel al contexto recuperado?**

Mide si las afirmaciones en la respuesta generada están **soportadas por los documentos recuperados**. No importa si la respuesta es "verdadera" en el mundo real, sino si está **alineada con el contexto dado**.

- Alta: Todo lo que dice la respuesta se puede respaldar con el contexto.
- Baja: El modelo "alucina" información que no está en los documentos.

### 2. `AnswerCorrectness()`
**¿La respuesta es correcta respecto a la verdad esperada?**

Compara la respuesta generada contra una **respuesta de referencia (ground truth)**. Combina exactitud semántica y factual.

- Alta: La respuesta coincide con la respuesta esperada.
- Baja: La respuesta es incorrecta o incompleta frente al ground truth.

### 3. `ResponseRelevancy()`
**¿La respuesta es relevante para la pregunta?**

Evalúa si la respuesta responde directamente a lo que se preguntó, sin desviarse ni incluir información innecesaria.

- Alta: La respuesta es concisa y responde exactamente la pregunta.
- Baja: La respuesta es vaga, evasiva o habla de temas no relacionados.

### 4. `FactualCorrectness()`
**¿Los hechos en la respuesta son correctos?**

Verifica la **precisión factual** de la respuesta comparándola contra el ground truth o fuentes externas. Se enfoca en hechos concretos (fechas, nombres, cifras, etc.).

- Alta: Los datos y hechos mencionados son correctos y verificables.
- Baja: Contiene errores factuales aunque suene coherente.


## Tabla resumen comparativo

| Métrica | ¿Qué evalúa? | ¿Necesita ground truth? | ¿Necesita contexto? |
|---|---|---|---|
| `Faithfulness` | Alucinaciones vs. contexto | No | Si |
| `AnswerCorrectness` | Calidad general vs. esperado | Si | No |
| `ResponseRelevancy` | Pertinencia a la pregunta | No | No |
| `FactualCorrectness` | Precisión de hechos concretos | Si | Si |

Basado en Kimothi (2025)

**Cómo mejorar métricas y sus trade-offs**

| Métrica | ¿Cómo mejorarla? | ¿Mejorarla afecta otras métricas? |
|---|---|---|
| `Faithfulness` | - Reducir chunk size para chunks más precisos y menos ruidosos - Aplicar reranker con mayor capacidad de entendimiento semántico - Prompt: "responde SOLO con lo que está en el contexto" - Usar LLM con menor tasa de alucinación - Chain-of-thought en el prompt - Few Shot Prompting | Puede bajar `ResponseRelevancy` (respuestas muy literales que no responden bien la pregunta) y `AnswerCorrectness` (si el contexto recuperado no contiene la respuesta completa) |
| `AnswerCorrectness` | - Mejorar calidad del ground truth - Mejorar el retrieval para que el contexto contenga toda la información necesaria - Usar LLMs más capaces - Query rewriting para preguntas ambiguas | Puede bajar `Faithfulness`: un LLM más capaz puede recurrir a conocimiento paramétrico fuera del contexto para completar la respuesta, introduciendo alucinaciones |
| `ResponseRelevancy` | - Prompt orientado a responder directamente sin parafrasear - Query rewriting antes del retrieval - Instrucciones de brevedad en el prompt - Mejorar el embedding model para mejor similitud semántica | Puede bajar `Faithfulness`: forzar respuestas directas y concisas puede llevar al LLM a omitir matices del contexto o añadir información no recuperada para completar la respuesta |
| `FactualCorrectness` | - Mejorar calidad y cobertura de los documentos fuente - Reranker - Chunking semántico para no partir hechos entre chunks - Verificación post-generación con modelo de fact-checking | Puede bajar `ResponseRelevancy` (respuestas más densas y técnicas que se desvían del intent) y `Faithfulness` (el modelo puede recurrir a conocimiento externo para "corregir" hechos que el contexto tiene incompletos) |

Tomado de Myriel (2024)

**Rango e interpretación de métricas según RAGAS (2025)**

**Faithfulness** — [docs](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/)

- Rango: [0, 1]
- Proporción de claims en la respuesta que pueden ser inferidos del contexto recuperado
- 0: ningún claim está respaldado por el contexto (alucinación total)
- 1: todos los claims están respaldados por el contexto


**Answer Correctness**

- Rango: [0, 1]
- Combinación ponderada de similitud semántica y factual correctness entre la respuesta y el ground truth
- 0: la respuesta no coincide semántica ni factualmente con el ground truth
- 1: la respuesta es idéntica al ground truth en semántica y hechos


**Response Relevancy**(Answer Relevancy) — [docs](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/)

- Si una respuesta aborda directamente la pregunta, entonces las preguntas que se pueden regenerar desde esa respuesta deberían parecerse mucho a la pregunta original
- Rango: [-1, 1] teórico por similitud coseno; en práctica [0, 1]
- Similitud coseno promedio entre la pregunta original y N preguntas regeneradas desde la respuesta
- 0: las preguntas regeneradas no se parecen a la pregunta original
- 1: la respuesta aborda directamente y de forma completa la pregunta

**Factual Correctness (mode=f1)** — [docs](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/factual_correctness/)

- Rango: [0, 1]
- F1 entre claims de la respuesta y claims del reference, calculado via precisión y recall
- 0: ningún claim de la respuesta coincide con el reference
- 1: coincidencia total entre los claims de la respuesta y el reference

In [ ]:
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_KEY"), temperature=0)
)
evaluator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-large", api_key=os.getenv("OPENAI_KEY"))
)

/var/folders/rp/m4lwqbf514n2ywy4y_md9txr0000gn/T/ipykernel_6316/945192752.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(
/var/folders/rp/m4lwqbf514n2ywy4y_md9txr0000gn/T/ipykernel_6316/945192752.py:4: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


In [43]:
lista_preguntas = [
    "¿Qué autonomía tiene el modelo que el CTO recomendó discontinuar?",
    "Qué versión de la app necesito para el modelo que reemplazó al LR-100",
    "Compré un LR-200 sin abrir hace 45 días, ¿puedo devolverlo y con qué cargo?",
    "Compré un LR-200 sin abrir en febrero de 2024, ¿puedo devolverlo y con qué cargo?",
    "Mi LR-200 muestra error E07 que no se va al reiniciar, ¿a quién llamo?",
    "Tengo garantía extendida y mi LR-200 no enciende, ¿qué línea uso?",
    "¿Qué modelo del mercado ofrece mejor garantía por menos de 600 €?",
    "Lista todos los bugs de software del modelo estrella que ya estén resueltos."    
]
lista_ground_truth = [
    "90 min",
    "v3.x",
    "reembolso íntegro sin cargo",
    "ya fuera de plazo.",
    "+34 900 111 222",
    "línea preferente",
    "LR-200",
    "F-301"    
]

res = rag_simple(lista_preguntas[0], collection_1)
print(res)

('No se especifica la autonomía del LR-100.', ['1. El CTO presenta el plan de simplificación de catálogo. Tras revisar las incidencias acumuladas en el último año, recomienda discontinuar el modelo LR-100 antes del cierre de año', 'Competidor B — modelo RoboNeat 5\n  Autonomía: 200 minutos\n  Mapeo: LiDAR combinado con cámara\n  Garantía: 24 meses\n  Precio: 699 €\n  Punto débil: app limitada, no soporta gestión multi-planta.'])


In [45]:
data = {
    "user_input":          [lista_preguntas[0]],
    "response":            [res[0]],
    "retrieved_contexts":  [res[1]],
    "reference":          [lista_ground_truth[0]]
}

dataset = Dataset.from_dict(data)

# Evaluar
results = evaluate(
    dataset=dataset,
        metrics=[
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        AnswerCorrectness(llm=evaluator_llm, embeddings=evaluator_embeddings),
        FactualCorrectness(llm=evaluator_llm),
    ],
)

display(results.to_pandas())

Evaluating: 100%|██████████| 4/4 [00:05<00:00,  1.47s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,answer_correctness,factual_correctness(mode=f1)
0,¿Qué autonomía tiene el modelo que el CTO reco...,[1. El CTO presenta el plan de simplificación ...,No se especifica la autonomía del LR-100.,90 min,0.0,0.0,0.056223,0.0


In [ ]:
## Limitaciones:
# abstention_correctness = 1.0  # si el ground truth no está en los chunks y el modelo se negó a responder
## no existe en RAGAS por defecto

## Notas adicionales

Benchmarks famosos para RAG (Kimothi, 2025):


**[SQuAD](https://rajpurkar.github.io/SQuAD-explorer/)** — Stanford Question Answering Dataset

Dataset de comprensión lectora con más de 100,000 preguntas sobre artículos de Wikipedia donde la respuesta es siempre un fragmento de texto del mismo párrafo. Se evalúa con Exact Match (EM), que verifica si la respuesta coincide exactamente con el ground truth, y F1-score, que mide el solapamiento parcial entre tokens. Es el benchmark más usado históricamente para QA general.

**[Natural Questions](https://ai.google.com/research/NaturalQuestions)** — Google

Preguntas reales extraídas de búsquedas de Google, respondidas con artículos de Wikipedia. A diferencia de SQuAD, las preguntas no fueron creadas mirando el contexto sino que provienen de usuarios reales, lo que las hace más representativas del mundo real.


**[HotpotQA](https://hotpotqa.github.io/)** — Multi-hop QA

Preguntas que requieren razonar sobre múltiples documentos simultáneamente para llegar a una respuesta. Multi-hop significa que no basta con recuperar un solo chunk, sino que hay que conectar información de dos o más fuentes. 

**[BEIR](https://github.com/beir-cellar/beir)** — Benchmark de Information Retrieval

Suite de 18 datasets heterogéneos que evalúa modelos de retrieval en dominios muy distintos (biomédico, legal, noticias, QA). Usa nDCG@10 (Normalized Discounted Cumulative Gain), una métrica que evalúa no solo si el documento correcto fue recuperado sino qué tan bien rankeado está entre los primeros 10 resultados.

**Multi-hop RAG** — HKUST

Dataset curado específicamente para RAG que requiere síntesis de múltiples fuentes. Similar a HotpotQA pero diseñado desde el principio para evaluar pipelines RAG completos, no solo modelos de comprensión lectora.


**[CRAG](https://github.com/facebookresearch/CRAG)** — Comprehensive RAG Benchmark (Meta, 2024)

El benchmark más completo Contiene 4,409 pares de preguntas sobre finanzas, deportes, música, cine y dominio abierto. Su característica distintiva es la evaluación en cuatro clases: perfect (respuesta correcta), acceptable (parcialmente correcta), missing (abstención correcta) y incorrect (alucinación). Es el único benchmark de la tabla que distingue explícitamente la abstención correcta como una categoría válida.

## Tarea en clase

Usar las métricas de RAGAS y comparar entre modelos con diferentes embeddings y contra nuestro simple Agentic RAG. ¿Puedes mejorar las métricas con un k o chunking razonable? ¿prompting?

In [ ]:
# To do

## Referencias

Myriel, D. (2024, November 24). Best practices in RAG evaluation: A comprehensive guide. Qdrant. https://qdrant.tech/blog/rag-evaluation-guide/

Ragas. (2025, December 9). List of available metrics. https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

Kimothi, A. (2025). A Simple Guide to Retrieval Augmented Generation. Simon and Schuster.